# 🥁 Drum OMR — MobileNetV3 Training

This notebook trains a model to read drum scores from bar images.

**What it learns:** Given a cropped image of one bar of drum notation,
output which drums are hit at each 16th-note position, and the note duration.

**Architecture:** MobileNetV3-Small (pretrained on ImageNet) + two prediction heads

**Output:**
- Drum head: 16 beat positions × 12 drums = 192 binary values
- Duration head: 16 beat positions × 10 duration classes = 160 values

**Final JSON per bar:**
```json
{ "beat": "1", "duration": "quarter", "drums": ["kick", "hi_hat_closed"] }
```

---
**Before running:** Upload `dataset.zip` to `MyDrive/drumhub/` in Google Drive.
Run `python ml/omr/prepare_upload.py` locally to create the zip.

In [ ]:
# ── 1. Install extra deps (torch/torchvision already on Colab) ────────────────
!pip install onnx onnxruntime onnxscript -q

In [ ]:
# ── 2. Mount Google Drive and unzip dataset ───────────────────────────────────
from google.colab import drive
drive.mount('/content/drive')

import zipfile, os

DRIVE_ROOT = '/content/drive/MyDrive/drumhub'
ZIP_PATH   = f'{DRIVE_ROOT}/dataset.zip'
DATA_ROOT  = '/content/dataset'
CKPT_DIR   = f'{DRIVE_ROOT}/checkpoints'

os.makedirs(CKPT_DIR, exist_ok=True)

if not os.path.exists(DATA_ROOT):
    print('Unzipping dataset...')
    with zipfile.ZipFile(ZIP_PATH, 'r') as z:
        z.extractall('/content')
    print('Done.')
else:
    print('Dataset already extracted.')

IMG_DIR = f'{DATA_ROOT}/images'
LBL_DIR = f'{DATA_ROOT}/labels'
print(f'Images: {len(os.listdir(IMG_DIR))}  Labels: {len(os.listdir(LBL_DIR))}')

In [ ]:
# ── 3. Config ─────────────────────────────────────────────────────────────────
import torch

# 16th-note grid: beat positions in 4/4 (1.0, 1.25, 1.5 ... 4.75)
BEAT_GRID = [round(1 + i * 0.25, 2) for i in range(16)]

DRUMS = [
    'hi_hat_closed', 'snare', 'kick',
    'ride', 'crash', 'hi_hat_open',
    'tom_low', 'tom_hi', 'hi_hat_pedal',
    'tom_mid', 'ride_bell', 'snare_rim',
]
DRUM_IDX = {d: i for i, d in enumerate(DRUMS)}

# Note durations the model will predict
DURATIONS = [
    'whole', 'half', 'dotted_quarter', 'quarter',
    'dotted_eighth', 'eighth', 'sixteenth', 'thirty_second',
    'triplet_eighth', 'triplet_sixteenth',
]
DUR_IDX = {d: i for i, d in enumerate(DURATIONS)}

N_BEATS     = len(BEAT_GRID)          # 16
N_DRUMS     = len(DRUMS)              # 12
N_DURATIONS = len(DURATIONS)          # 10
N_DRUM_OUT  = N_BEATS * N_DRUMS       # 192  — which drums are hit
N_DUR_OUT   = N_BEATS * N_DURATIONS   # 160  — duration at each beat position

IMG_H, IMG_W    = 128, 384
BATCH_SIZE      = 32
EPOCHS          = 15    # Phase 1: best checkpoint was epoch 8 last run
FINETUNE_EPOCHS = 25    # Phase 2: still improving at epoch 15 last run
LR              = 3e-4
DUR_WEIGHT      = 0.5   # weight of duration loss relative to drum loss
THRESHOLD       = 0.5

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {DEVICE}')
print(f'Drum output: {N_DRUM_OUT}  Duration output: {N_DUR_OUT}')

In [ ]:
# ── 4. Dataset class ──────────────────────────────────────────────────────────
import json
import numpy as np
from pathlib import Path
from PIL import Image
import torch
from torch.utils.data import Dataset
import torchvision.transforms as T


def normalize_duration(dur_str: str) -> int:
    """Map a duration string to a DURATIONS index, with graceful fallback."""
    if dur_str in DUR_IDX:
        return DUR_IDX[dur_str]
    # Partial match for combos like 'dotted_triplet_eighth', 'sextuplet_eighth'
    for dur in DURATIONS:
        if dur in dur_str:
            return DUR_IDX[dur]
    return DUR_IDX['quarter']  # safe default


def label_to_target(label_path: str) -> tuple[torch.Tensor, torch.Tensor]:
    """Convert a label JSON to two targets:
    y_drums : (192,) float — which drums are hit at each 16th-note position
    y_dur   : (16,)  long  — duration class index at each beat position
    """
    data    = json.loads(Path(label_path).read_text())
    y_drums = torch.zeros(N_BEATS, N_DRUMS)
    y_dur   = torch.zeros(N_BEATS, dtype=torch.long)   # default: quarter

    for beat_entry in data.get('beats', []):
        if beat_entry.get('rest'):
            continue
        beat_val = float(beat_entry['beat'])
        dists    = [abs(beat_val - g) for g in BEAT_GRID]
        beat_idx = int(np.argmin(dists))
        if min(dists) > 0.13:   # too far from 16th-note grid — skip
            continue

        y_dur[beat_idx] = normalize_duration(beat_entry.get('duration', 'quarter'))

        for drum in beat_entry.get('drums', []):
            if drum in DRUM_IDX:
                y_drums[beat_idx, DRUM_IDX[drum]] = 1.0

    return y_drums.flatten(), y_dur   # (192,), (16,)


class DrumBarDataset(Dataset):
    def __init__(self, image_paths: list, label_paths: list, augment: bool = False):
        self.images = image_paths
        self.labels = label_paths

        base = [
            T.Resize((IMG_H, IMG_W)),
            T.Grayscale(num_output_channels=3),
            T.ToTensor(),
            T.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
        ]
        aug = [
            T.Resize((IMG_H, IMG_W)),
            T.Grayscale(num_output_channels=3),
            T.RandomAffine(degrees=2, translate=(0, 0.04)),
            T.ColorJitter(brightness=0.4, contrast=0.4),
            T.GaussianBlur(kernel_size=3, sigma=(0.1, 1.5)),
            T.ToTensor(),
            T.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
        ]
        self.transform = T.Compose(aug if augment else base)

    def __len__(self):
        return len(self.images)

    def __getitem__(self, idx):
        img            = Image.open(self.images[idx]).convert('L')
        x              = self.transform(img)
        y_drums, y_dur = label_to_target(self.labels[idx])
        return x, y_drums, y_dur

print('Dataset class defined.')

In [ ]:
# ── 5. Train / val / test split (by song, not by bar) ────────────────────────
import random
from collections import defaultdict

random.seed(42)

img_files = sorted(Path(IMG_DIR).glob('*.png'))
lbl_files = {p.stem: str(p) for p in Path(LBL_DIR).glob('*.json')}

songs = defaultdict(list)
for img in img_files:
    stem = img.stem
    song = stem[:stem.rfind('_bar')]
    if stem in lbl_files:
        songs[song].append((str(img), lbl_files[stem]))

song_names = list(songs.keys())
random.shuffle(song_names)

n_val  = max(1, int(len(song_names) * 0.10))
n_test = max(1, int(len(song_names) * 0.10))

test_songs  = set(song_names[:n_test])
val_songs   = set(song_names[n_test:n_test + n_val])
train_songs = set(song_names[n_test + n_val:])

def collect(song_set):
    imgs, lbls = [], []
    for s in song_set:
        for img, lbl in songs[s]:
            imgs.append(img)
            lbls.append(lbl)
    return imgs, lbls

train_imgs, train_lbls = collect(train_songs)
val_imgs,   val_lbls   = collect(val_songs)
test_imgs,  test_lbls  = collect(test_songs)

print(f'Songs  — train: {len(train_songs)}  val: {len(val_songs)}  test: {len(test_songs)}')
print(f'Bars   — train: {len(train_imgs)}   val: {len(val_imgs)}   test: {len(test_imgs)}')

In [ ]:
# ── 6. Data loaders ───────────────────────────────────────────────────────────
from torch.utils.data import DataLoader

train_ds = DrumBarDataset(train_imgs, train_lbls, augment=True)
val_ds   = DrumBarDataset(val_imgs,   val_lbls,   augment=False)
test_ds  = DrumBarDataset(test_imgs,  test_lbls,  augment=False)

train_dl = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,  num_workers=2, pin_memory=True)
val_dl   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)
test_dl  = DataLoader(test_ds,  batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)

x, y_drums, y_dur = next(iter(train_dl))
print(f'Batch — images: {x.shape}  drums: {y_drums.shape}  durations: {y_dur.shape}  active hits: {y_drums.sum().item():.0f}')

In [ ]:
# ── 7. Visualise one sample ───────────────────────────────────────────────────
import matplotlib.pyplot as plt

img_raw        = Image.open(train_imgs[0]).convert('L')
y_drums, y_dur = label_to_target(train_lbls[0])
target         = y_drums.reshape(N_BEATS, N_DRUMS).numpy()

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 4))

ax1.imshow(img_raw, cmap='gray')
ax1.set_title(Path(train_imgs[0]).stem)
ax1.axis('off')

ax2.imshow(target.T, aspect='auto', cmap='Blues', vmin=0, vmax=1)
ax2.set_xticks(range(N_BEATS))
ax2.set_xticklabels([str(b) for b in BEAT_GRID], rotation=45, fontsize=8)
ax2.set_yticks(range(N_DRUMS))
ax2.set_yticklabels(DRUMS, fontsize=8)
ax2.set_title('Target grid (blue = hit)')
ax2.set_xlabel('Beat position')

plt.tight_layout()
plt.show()

print('\nBeat   Duration            Drums')
print('-' * 55)
for bi, beat_val in enumerate(BEAT_GRID):
    if target[bi].sum() > 0:
        drums_hit = [DRUMS[di] for di in range(N_DRUMS) if target[bi, di] > 0]
        dur       = DURATIONS[y_dur[bi].item()]
        print(f'{beat_val:<7} {dur:<20} {drums_hit}')

In [ ]:
# ── 8. Model — two-head MobileNetV3 ──────────────────────────────────────────
#
# Architecture:
#   Image (3×128×384)
#     → MobileNetV3 backbone (frozen in Phase 1)
#     → global avg pool → (576,)
#     → shared: Linear(576→1024) + Hardswish + Dropout
#     ├─→ drum_head:  Linear(1024→192)  sigmoid → which drums are hit
#     └─→ dur_head:   Linear(1024→160)  softmax → duration class per beat

import torch.nn as nn
from torchvision.models import mobilenet_v3_small, MobileNet_V3_Small_Weights


class DrumOMRModel(nn.Module):
    def __init__(self):
        super().__init__()
        base = mobilenet_v3_small(weights=MobileNet_V3_Small_Weights.DEFAULT)

        self.features = base.features              # CNN backbone
        self.avgpool  = base.avgpool               # global average pool
        self.shared   = nn.Sequential(             # Linear(576→1024) + Hardswish + Dropout
            *list(base.classifier[:-1])
        )
        feat_size = base.classifier[-1].in_features  # 1024
        self.drum_head = nn.Linear(feat_size, N_DRUM_OUT)   # → 192
        self.dur_head  = nn.Linear(feat_size, N_DUR_OUT)    # → 160

    def forward(self, x):
        x = self.features(x)
        x = self.avgpool(x)
        x = torch.flatten(x, 1)
        x = self.shared(x)
        return self.drum_head(x), self.dur_head(x)


model = DrumOMRModel().to(DEVICE)

# Freeze backbone for Phase 1 — only shared + heads are trainable
for param in model.features.parameters():
    param.requires_grad = False

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total     = sum(p.numel() for p in model.parameters())
print(f'Parameters — total: {total:,}  trainable: {trainable:,}  frozen: {total - trainable:,}')

In [ ]:
# ── 9. Training loop ──────────────────────────────────────────────────────────
from torch.optim import AdamW
from torch.optim.lr_scheduler import CosineAnnealingLR

# pos_weight: corrects class imbalance for rare drums
all_drum_targets = torch.stack([label_to_target(l)[0] for l in train_lbls])
pos        = all_drum_targets.sum(0).clamp(min=1)
neg        = (len(train_lbls) - all_drum_targets.sum(0)).clamp(min=1)
pos_weight = (neg / pos).to(DEVICE)
print(f'pos_weight range: {pos_weight.min():.1f} – {pos_weight.max():.1f}')

drum_criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
dur_criterion  = nn.CrossEntropyLoss()
optimizer      = AdamW(filter(lambda p: p.requires_grad, model.parameters()), lr=LR, weight_decay=1e-4)
scheduler      = CosineAnnealingLR(optimizer, T_max=EPOCHS)


def run_epoch(loader, train: bool, opt=None):
    if opt is None:
        opt = optimizer
    model.train(train)
    total_loss, drum_correct, total = 0.0, 0, 0

    with torch.set_grad_enabled(train):
        for x, y_drums, y_dur in loader:
            x, y_drums, y_dur = x.to(DEVICE), y_drums.to(DEVICE), y_dur.to(DEVICE)
            B = x.size(0)

            drum_logits, dur_logits = model(x)

            # Drum loss: multi-label binary classification
            drum_loss = drum_criterion(drum_logits, y_drums)

            # Duration loss: only at beat positions that actually have hits
            hit_mask = y_drums.view(B, N_BEATS, N_DRUMS).sum(dim=2) > 0  # (B, N_BEATS)
            if hit_mask.sum() > 0:
                dur_loss = dur_criterion(
                    dur_logits.view(B, N_BEATS, N_DURATIONS)[hit_mask],
                    y_dur[hit_mask],
                )
            else:
                dur_loss = torch.tensor(0.0, device=DEVICE)

            loss = drum_loss + DUR_WEIGHT * dur_loss

            if train:
                opt.zero_grad()
                loss.backward()
                opt.step()

            total_loss  += loss.item() * B
            preds        = (drum_logits.sigmoid() > THRESHOLD).float()
            drum_correct += (preds == y_drums).all(dim=1).sum().item()
            total        += B

    return total_loss / total, drum_correct / total


history = {'train_loss': [], 'val_loss': [], 'train_acc': [], 'val_acc': []}
best_val_loss = float('inf')
CKPT_PATH = f'{CKPT_DIR}/omr_best.pt'

for epoch in range(1, EPOCHS + 1):
    tr_loss, tr_acc = run_epoch(train_dl, train=True)
    va_loss, va_acc = run_epoch(val_dl,   train=False)
    scheduler.step()

    history['train_loss'].append(tr_loss)
    history['val_loss'].append(va_loss)
    history['train_acc'].append(tr_acc)
    history['val_acc'].append(va_acc)

    if va_loss < best_val_loss:
        best_val_loss = va_loss
        torch.save(model.state_dict(), CKPT_PATH)
        saved = ' ← saved'
    else:
        saved = ''

    print(f'Epoch {epoch:2d}/{EPOCHS}  '
          f'train loss={tr_loss:.4f} acc={tr_acc:.3f}  '
          f'val loss={va_loss:.4f} acc={va_acc:.3f}{saved}')

In [ ]:
# ── 10. Plot Phase 1 training curves ──────────────────────────────────────────
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

ax1.plot(history['train_loss'], label='train')
ax1.plot(history['val_loss'],   label='val')
ax1.set_title('Phase 1 Loss'); ax1.set_xlabel('Epoch'); ax1.legend()

ax2.plot(history['train_acc'], label='train')
ax2.plot(history['val_acc'],   label='val')
ax2.set_title('Phase 1 Exact-bar accuracy'); ax2.set_xlabel('Epoch'); ax2.legend()

plt.tight_layout()
plt.show()

In [ ]:
# ── 11. Evaluate Phase 1 on held-out test set ────────────────────────────────
from sklearn.metrics import f1_score

model.load_state_dict(torch.load(CKPT_PATH))
model.eval()

all_drum_preds, all_drum_gts = [], []
all_dur_preds,  all_dur_gts  = [], []

with torch.no_grad():
    for x, y_drums, y_dur in test_dl:
        drum_logits, dur_logits = model(x.to(DEVICE))

        drum_preds = (drum_logits.sigmoid() > THRESHOLD).cpu().float()
        dur_preds  = dur_logits.view(-1, N_BEATS, N_DURATIONS).argmax(dim=2).cpu()

        all_drum_preds.append(drum_preds)
        all_drum_gts.append(y_drums)
        all_dur_preds.append(dur_preds)
        all_dur_gts.append(y_dur)

P      = torch.cat(all_drum_preds).numpy()
GT     = torch.cat(all_drum_gts).numpy()
P_dur  = torch.cat(all_dur_preds)
GT_dur = torch.cat(all_dur_gts)

exact_acc = (P == GT).all(axis=1).mean()
cell_acc  = (P == GT).mean()
print(f'Test exact-bar accuracy : {exact_acc:.3f} ({exact_acc*100:.1f}%)')
print(f'Test cell accuracy      : {cell_acc:.3f} ({cell_acc*100:.1f}%)')
print()

P_drums  = P.reshape(-1, N_BEATS, N_DRUMS).max(axis=1)
GT_drums = GT.reshape(-1, N_BEATS, N_DRUMS).max(axis=1)

print(f'{"Drum":<18} {"F1":>6}')
print('-' * 26)
for i, drum in enumerate(DRUMS):
    f1 = f1_score(GT_drums[:, i], P_drums[:, i], zero_division=0)
    print(f'{drum:<18} {f1:6.3f}')

print()
hit_mask = GT.reshape(-1, N_BEATS, N_DRUMS).sum(axis=2) > 0
if hit_mask.sum() > 0:
    dur_acc = (P_dur.numpy()[hit_mask] == GT_dur.numpy()[hit_mask]).mean()
    print(f'Duration accuracy (at hit positions): {dur_acc:.3f} ({dur_acc*100:.1f}%)')
    print()
    gt_dur_flat = GT_dur.numpy()[hit_mask]
    print(f'{"Duration":<20} {"Count":>6}  Acc')
    print('-' * 36)
    for i, dur in enumerate(DURATIONS):
        count = (gt_dur_flat == i).sum()
        if count > 0:
            correct = ((P_dur.numpy()[hit_mask] == i) & (gt_dur_flat == i)).sum()
            print(f'{dur:<20} {count:>6}  {correct/count:.2f}')

In [ ]:
# ── 12. Visualise predictions ─────────────────────────────────────────────────
n_show = 4
fig, axes = plt.subplots(n_show, 3, figsize=(15, n_show * 3))

model.eval()
with torch.no_grad():
    for i in range(n_show):
        img_raw = Image.open(test_imgs[i]).convert('L')
        x, y_drums, y_dur = test_ds[i]
        drum_logits, dur_logits = model(x.unsqueeze(0).to(DEVICE))

        pred  = (drum_logits.sigmoid() > THRESHOLD).cpu().float().reshape(N_BEATS, N_DRUMS)
        truth = y_drums.reshape(N_BEATS, N_DRUMS)

        axes[i, 0].imshow(img_raw, cmap='gray'); axes[i, 0].axis('off')
        axes[i, 0].set_title(Path(test_imgs[i]).stem[:40], fontsize=8)

        axes[i, 1].imshow(truth.numpy().T, aspect='auto', cmap='Blues', vmin=0, vmax=1)
        axes[i, 1].set_title('Ground truth')
        axes[i, 1].set_yticks(range(N_DRUMS)); axes[i, 1].set_yticklabels(DRUMS, fontsize=7)

        axes[i, 2].imshow(pred.numpy().T, aspect='auto', cmap='Oranges', vmin=0, vmax=1)
        axes[i, 2].set_title('Predicted')
        axes[i, 2].set_yticks(range(N_DRUMS)); axes[i, 2].set_yticklabels(DRUMS, fontsize=7)

plt.tight_layout()
plt.show()

In [ ]:
# ── 13. Unfreeze backbone and fine-tune everything (Phase 2) ──────────────────
# Start from best Phase 1 checkpoint, unfreeze all layers, train at LR/10.
# This lets the backbone adapt its features specifically to drum notation.

model.load_state_dict(torch.load(CKPT_PATH))

for param in model.parameters():
    param.requires_grad = True

trainable2 = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'All {trainable2:,} parameters now trainable')

CKPT_PATH_FT = f'{CKPT_DIR}/omr_finetuned.pt'
optimizer2   = AdamW(model.parameters(), lr=LR / 10, weight_decay=1e-4)
scheduler2   = CosineAnnealingLR(optimizer2, T_max=FINETUNE_EPOCHS)

history2 = {'train_loss': [], 'val_loss': [], 'train_acc': [], 'val_acc': []}
best_ft_loss = float('inf')

for epoch in range(1, FINETUNE_EPOCHS + 1):
    tr_loss, tr_acc = run_epoch(train_dl, train=True,  opt=optimizer2)
    va_loss, va_acc = run_epoch(val_dl,   train=False)
    scheduler2.step()

    history2['train_loss'].append(tr_loss)
    history2['val_loss'].append(va_loss)
    history2['train_acc'].append(tr_acc)
    history2['val_acc'].append(va_acc)

    if va_loss < best_ft_loss:
        best_ft_loss = va_loss
        torch.save(model.state_dict(), CKPT_PATH_FT)
        saved = ' ← saved'
    else:
        saved = ''

    print(f'Finetune {epoch:2d}/{FINETUNE_EPOCHS}  '
          f'train loss={tr_loss:.4f} acc={tr_acc:.3f}  '
          f'val loss={va_loss:.4f} acc={va_acc:.3f}{saved}')

CKPT_PATH = CKPT_PATH_FT
print(f'\nBest finetuned model: {CKPT_PATH}')

In [ ]:
# ── 13a. Plot Phase 2 training curves ─────────────────────────────────────────
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

ax1.plot(history2['train_loss'], label='train')
ax1.plot(history2['val_loss'],   label='val')
ax1.set_title('Phase 2 Loss'); ax1.set_xlabel('Epoch'); ax1.legend()

ax2.plot(history2['train_acc'], label='train')
ax2.plot(history2['val_acc'],   label='val')
ax2.set_title('Phase 2 Exact-bar accuracy'); ax2.set_xlabel('Epoch'); ax2.legend()

plt.tight_layout()
plt.show()

In [ ]:
# ── 13b. Re-evaluate after Phase 2 ───────────────────────────────────────────
model.load_state_dict(torch.load(CKPT_PATH))
model.eval()

all_drum_preds, all_drum_gts = [], []
all_dur_preds,  all_dur_gts  = [], []

with torch.no_grad():
    for x, y_drums, y_dur in test_dl:
        drum_logits, dur_logits = model(x.to(DEVICE))

        drum_preds = (drum_logits.sigmoid() > THRESHOLD).cpu().float()
        dur_preds  = dur_logits.view(-1, N_BEATS, N_DURATIONS).argmax(dim=2).cpu()

        all_drum_preds.append(drum_preds)
        all_drum_gts.append(y_drums)
        all_dur_preds.append(dur_preds)
        all_dur_gts.append(y_dur)

P      = torch.cat(all_drum_preds).numpy()
GT     = torch.cat(all_drum_gts).numpy()
P_dur  = torch.cat(all_dur_preds)
GT_dur = torch.cat(all_dur_gts)

exact_acc = (P == GT).all(axis=1).mean()
cell_acc  = (P == GT).mean()
print(f'Test exact-bar accuracy : {exact_acc:.3f} ({exact_acc*100:.1f}%)')
print(f'Test cell accuracy      : {cell_acc:.3f} ({cell_acc*100:.1f}%)')
print()

P_drums  = P.reshape(-1, N_BEATS, N_DRUMS).max(axis=1)
GT_drums = GT.reshape(-1, N_BEATS, N_DRUMS).max(axis=1)

print(f'{"Drum":<18} {"F1":>6}')
print('-' * 26)
for i, drum in enumerate(DRUMS):
    f1 = f1_score(GT_drums[:, i], P_drums[:, i], zero_division=0)
    print(f'{drum:<18} {f1:6.3f}')

print()
hit_mask = GT.reshape(-1, N_BEATS, N_DRUMS).sum(axis=2) > 0
if hit_mask.sum() > 0:
    dur_acc = (P_dur.numpy()[hit_mask] == GT_dur.numpy()[hit_mask]).mean()
    print(f'Duration accuracy (at hit positions): {dur_acc:.3f} ({dur_acc*100:.1f}%)')
    print()
    gt_dur_flat = GT_dur.numpy()[hit_mask]
    print(f'{"Duration":<20} {"Count":>6}  Acc')
    print('-' * 36)
    for i, dur in enumerate(DURATIONS):
        count = (gt_dur_flat == i).sum()
        if count > 0:
            correct = ((P_dur.numpy()[hit_mask] == i) & (gt_dur_flat == i)).sum()
            print(f'{dur:<20} {count:>6}  {correct/count:.2f}')

In [ ]:
# ── 14. Export to ONNX ────────────────────────────────────────────────────────
import onnx
import onnxruntime as ort

ONNX_PATH = f'{CKPT_DIR}/omr.onnx'

model.load_state_dict(torch.load(CKPT_PATH))
model.eval()

dummy_input = torch.randn(1, 3, IMG_H, IMG_W).to(DEVICE)

torch.onnx.export(
    model,
    dummy_input,
    ONNX_PATH,
    input_names=['image'],
    output_names=['drum_logits', 'dur_logits'],
    dynamic_axes={
        'image':       {0: 'batch'},
        'drum_logits': {0: 'batch'},
        'dur_logits':  {0: 'batch'},
    },
    opset_version=18,
)

sess     = ort.InferenceSession(ONNX_PATH)
dummy_np = dummy_input.cpu().numpy()
ort_drum, ort_dur = sess.run(None, {'image': dummy_np})
with torch.no_grad():
    pt_drum, pt_dur = model(dummy_input)

drum_diff = abs(ort_drum - pt_drum.cpu().numpy()).max()
dur_diff  = abs(ort_dur  - pt_dur.cpu().numpy()).max()
print(f'ONNX exported to: {ONNX_PATH}')
print(f'Max diff — drums: {drum_diff:.2e}  duration: {dur_diff:.2e}')

In [ ]:
# ── 15. Decode ONNX output back to JSON ───────────────────────────────────────
# This is the function you will call in Electron / FastAPI:
#   image → ONNX inference → JSON with beat + duration + drums

import torchvision.transforms as transforms

def predict_bar(image_path: str) -> dict:
    """Run ONNX inference on one bar image.
    Returns a dict matching the label format: beat, duration, drums."""
    transform = transforms.Compose([
        transforms.Resize((IMG_H, IMG_W)),
        transforms.Grayscale(num_output_channels=3),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
    ])

    img  = Image.open(image_path).convert('L')
    x    = transform(img).unsqueeze(0).numpy()              # (1, 3, H, W)

    drum_logits, dur_logits = sess.run(None, {'image': x})

    drum_probs = 1 / (1 + np.exp(-drum_logits[0]))          # sigmoid → (192,)
    drum_grid  = drum_probs.reshape(N_BEATS, N_DRUMS)

    dur_grid   = dur_logits[0].reshape(N_BEATS, N_DURATIONS)
    dur_preds  = dur_grid.argmax(axis=1)                    # (16,) class indices

    beats = []
    for bi, beat_val in enumerate(BEAT_GRID):
        drums = [DRUMS[di] for di in range(N_DRUMS) if drum_grid[bi, di] > THRESHOLD]
        if drums:
            beats.append({
                'beat':     str(beat_val),
                'duration': DURATIONS[dur_preds[bi]],
                'drums':    drums,
            })

    return {'time_signature': '4/4', 'beats': beats}


# Test on a few samples to verify output format
for idx in range(min(3, len(test_imgs))):
    result = predict_bar(test_imgs[idx])
    if result['beats']:
        print(f'--- {Path(test_imgs[idx]).stem} ---')
        print(json.dumps(result, indent=2))
        break

## Next steps

1. Download `omr.onnx` from Drive to your M1
2. Load it in FastAPI: `onnxruntime.InferenceSession('omr.onnx')`
3. Electron sends a bar image → FastAPI runs inference → returns JSON (beat + duration + drums) → VexFlow renders it
4. To improve weak cymbals (crash, ride bell, open hi-hat): add more GP5 songs and retrain
5. For triplets: extend BEAT_GRID to 32 positions (32nd-note grid) to capture triplet beat positions